# Testing frameworks: `pytest`

[Documentation](https://docs.pytest.org/en/6.2.x/)

`pytest` is a 3rd party library built as an alternative to Standard Library's `unittest`.

`pytest` supports execution of unittest test cases, but the real advantage of `pytest` comes by writing `pytest` test cases. `pytest` test cases are a series of functions in a Python file starting with the name `test_`.

`pytest` has some other great features:

* Tests are expressive and readable — no boilerplate code required
* Support for the built-in `assert` statement instead of using special `self.assert*()` methods
* Support for filtering for test cases
* Ability to rerun from the last failing test
* Marks and parametrized tests
* Modular fixtures
* An ecosystem of hundreds of plugins to extend the functionality

### Installation

Because it is a 3rd party library, you should install it using `pip`:

```
pip install pytest
```

### Writing a simple test

Tests in `pytest` are simple functions. Assertions are done using `assert` statement.

In order to execute pytests inside Jupyter Notebook, we're going to use a package called `ipytest`.

In [2]:
import sys                                                                                                                                                                                                  
!{sys.executable} -m pip install pytest
!{sys.executable} -m pip install ipytest

import ipytest
ipytest.autoconfig()

You should consider upgrading via the '/Users/iulia/.virtualenvs/python-testing/bin/python -m pip install --upgrade pip' command.
You should consider upgrading via the '/Users/iulia/.virtualenvs/python-testing/bin/python -m pip install --upgrade pip' command.


In [4]:
%%ipytest -qq


def func(x):
    return x + 1


def test_func_pass():
    assert func(3) == 4
    
def test_func_fail():
    assert func(3) == 5

.F                                                                                           [100%]
============================================= FAILURES =============================================
__________________________________________ test_func_fail __________________________________________

    def test_func_fail():
>       assert func(3) == 5
E       assert 4 == 5
E        +  where 4 = func(3)

/var/folders/s2/_752pzb50tg00mv9ggtr6p8c0000gn/T/ipykernel_67066/2446215452.py:9: AssertionError
===================================== short test summary info ======================================
FAILED tmpi6944lzh.py::test_func_fail - assert 4 == 5


### Assert that an exception is raised

`pytest` implements a helper function that can be used with the `with` statement:

In [14]:
%%ipytest -qq

import pytest


def func():
    raise ValueError


def test_raises():
    with pytest.raises(ValueError):
        func()

.                                                                                            [100%]


## Command-line interface

`pytest`'s command line interface is more powerful than `unittest`s.

Running `pytest --help` will give you a list of all the arguments throught which the behaviour of the testing framework can be adjusted.

In [15]:
!{sys.executable} -m pytest --help



usage: __main__.py [options] [file_or_dir] [file_or_dir] [...]

positional arguments:
  file_or_dir

general:
  -k EXPRESSION         only run tests which match the given substring
                        expression. An expression is a python evaluatable
                        expression where all names are substring-matched against
                        test names and their parent classes. Example: -k
                        'test_method or test_other' matches all test functions
                        and classes whose name contains 'test_method' or
                        'test_other', while -k 'not test_method' matches those
                        that don't contain 'test_method' in their names. -k 'not
                        test_method and not test_other' will eliminate the
                        matches. Additionally keywords are matched to classes
                        and functions containing extra names in their
                        'extra_keyword_matches' set, as 

### Invocation

You can invoke testing through the Python interpreter from the command line:

```
python -m pytest [...]
```

This is almost equivalent to invoking the command line script `pytest [...]` directly, except that calling via python will also add the current directory to `sys.path`.

### Running specific tests

Pytest supports several ways to run and select tests from the command-line.

Run tests in a module
```
pytest test_mod.py
```
Run tests in a directory
```
pytest testing/
```
Run tests by keyword expressions
```
pytest -k "MyClass and not method"
```
This will run tests which contain names that match the given string expression (case-insensitive), which can include Python operators that use filenames, class names and function names as variables. The example above will run `TestMyClass.test_something` but not `TestMyClass.test_method_simple`.

#### Run tests by node ids

Each collected test is assigned a unique nodeid which consist of the module filename followed by specifiers like class names, function names and parameters from parametrization, separated by :: characters.

To run a specific test within a module:

```
pytest test_mod.py::test_func
```
Another example specifying a test method in the command line:
```
pytest test_mod.py::TestClass::test_method
```

### Stopping on failures

To stop the testing process after the first (N) failures:

```
pytest -x           # stop after first failure
pytest --maxfail=2  # stop after two failures
```

Read more about `pytest`'s command-line interface [here](https://docs.pytest.org/en/6.2.x/usage.html).

## Fixtures

`pytest` fixtures offer dramatic improvements over the classic xUnit style of setup/teardown functions:

* fixtures have explicit names and are activated by declaring their use from test functions, modules, classes or whole projects.
* fixtures are implemented in a modular manner, as each fixture name triggers a fixture function which can itself use other fixtures.
* fixture management scales from simple unit to complex functional testing, allowing to parametrize fixtures and tests according to configuration and component options, or to re-use fixtures across function, class, module or whole test session scopes.
* teardown logic can be easily, and safely managed, no matter how many fixtures are used, without the need to carefully handle errors by hand or micromanage the order that cleanup steps are added.

We can tell pytest that a particular function is a fixture by decorating it with `@pytest.fixture`.

In [16]:
%%ipytest -qq


import pytest


class Fruit:
    def __init__(self, name):
        self.name = name

    def __eq__(self, other):
        return self.name == other.name


@pytest.fixture
def my_fruit():
    return Fruit("apple")


@pytest.fixture
def fruit_basket(my_fruit):
    return [Fruit("banana"), my_fruit]


def test_my_fruit_in_basket(my_fruit, fruit_basket):
    assert my_fruit in fruit_basket

.                                                                                            [100%]


## Marks

By using the `pytest.mark` helper you can easily set metadata on your test functions. Markers can be built-in or user-defined. You can list all the markers, including built-in and custom, using the CLI `pytest --markers`.

Here are some of the builtin markers:

* `usefixtures` - use fixtures on a test function or class
* `filterwarnings` - filter certain warnings of a test function
* `skip` - always skip a test function
* `skipif` - skip a test function if a certain condition is met
* `xfail` - produce an “expected failure” outcome if a certain condition is met
* `parametrize` - perform multiple calls to the same test function.

It’s easy to create custom markers or to apply markers to whole test classes or modules. Those markers can be used by plugins, and also are commonly used to select tests on the command-line with the -m option.

### Registering marks

You can register custom marks in your `pytest.ini` file like this:

```ini
[pytest]
markers =
    slow: marks tests as slow (deselect with '-m "not slow"')
    serial
```

### Marking tests

In [21]:
%%ipytest -qq -m slow


import pytest


def pytest_configure(config):
    config.addinivalue_line(
        "markers", "slow: marks tests as slow"
    )

def func():
    pass


@pytest.mark.slow
def test_func():
    assert func() is None

def test_serial():
    assert 1 == 0

.                                                                                            [100%]


## Parametrized tests

The builtin `pytest.mark.parametrize` decorator enables parametrization of arguments for a test function. Here is a typical example of a test function that implements checking that a certain input leads to an expected output:

In [24]:
%%ipytest -qq


import pytest


@pytest.mark.parametrize("test_input,expected", [("3+5", 8), ("2+4", 6), ("6*9", 42)])
def test_eval(test_input, expected):
    assert eval(test_input) == expected

..F                                                                                          [100%]
============================================= FAILURES =============================================
________________________________________ test_eval[6*9-42] _________________________________________

test_input = '6*9', expected = 42

    @pytest.mark.parametrize("test_input,expected", [("3+5", 8), ("2+4", 6), ("6*9", 42)])
    def test_eval(test_input, expected):
>       assert eval(test_input) == expected
E       AssertionError: assert 54 == 42
E        +  where 54 = eval('6*9')

/var/folders/s2/_752pzb50tg00mv9ggtr6p8c0000gn/T/ipykernel_55685/2167493762.py:6: AssertionError
===================================== short test summary info ======================================
FAILED tmpif757s36.py::test_eval[6*9-42] - AssertionError: assert 54 == 42
